In [3]:
import pandas as pd
from dotenv import load_dotenv
import os
import plotly.express as px
from plotly.subplots import make_subplots

load_dotenv()

True

## Chargement données de consommation électrique en France entre 2020 et 2025

In [4]:
df = pd.read_csv(os.getenv("DATA_STORAGE") + 'power_consumption_meteo_2021_2025_utc.csv', parse_dates=['start_date', 'end_date'])
print(df.shape)
df.head()

(43074, 10)


,start_date,end_date,value,mean,min,max,median,std,q1,q3
0,2021-01-01 00:00:00,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 01:00:00,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 02:00:00,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 03:00:00,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 04:00:00,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


In [5]:
df.dropna(inplace=True)
df.shape

(43069, 10)

In [6]:
print(df.describe())

                          start_date                       end_date  \
count                          43069                          43069   
mean   2023-06-17 10:15:11.848429056  2023-06-17 11:15:11.848429056   
min              2021-01-01 00:00:00            2021-01-01 01:00:00   
25%              2022-03-25 17:00:00            2022-03-25 18:00:00   
50%              2023-06-17 10:00:00            2023-06-17 11:00:00   
75%              2024-09-08 03:00:00            2024-09-08 04:00:00   
max              2025-11-30 22:00:00            2025-11-30 23:00:00   
std                              NaN                            NaN   

              value          mean           min           max        median  \
count  43069.000000  43069.000000  43069.000000  43069.000000  43069.000000   
mean   50794.203778     13.075924      8.631401     18.952602     12.743789   
min    30040.500000     -3.700000    -11.700000      1.800000     -5.400000   
25%    42779.000000      7.946000      3.400

In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 43069 entries, 0 to 43073
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   start_date  43069 non-null  datetime64[ns]
 1   end_date    43069 non-null  datetime64[ns]
 2   value       43069 non-null  float64       
 3   mean        43069 non-null  float64       
 4   min         43069 non-null  float64       
 5   max         43069 non-null  float64       
 6   median      43069 non-null  float64       
 7   std         43069 non-null  float64       
 8   q1          43069 non-null  float64       
 9   q3          43069 non-null  float64       
dtypes: datetime64[ns](2), float64(8)
memory usage: 3.6 MB
None


### Consommation horaire de puissance électrique en France (2021-2025)

In [8]:
fig = px.line(df, x='end_date', y='value', title="Consommation horaire de puissance électrique en France (2021-2025)",
    labels={'end_date': 'Date', 'value': 'Consommation (MW)'})
fig.show()

In [16]:
fig_2rows = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Consommation de puissance électrique (MW)", "Température (°C)"),
    vertical_spacing=0.1
)
fig_rte = px.line(df, x='end_date', y='value')
fig_2rows.add_trace(fig_rte.data[0], row=1, col=1)

fig_temp = px.line(df, x='end_date', y='mean')
fig_temp.data[0].line.color = 'red'
fig_2rows.add_trace(fig_temp.data[0], row=2, col=1)

fig_2rows.update_layout(height=600)
fig_2rows.show()

In [47]:
# comparaison consommation par an
df_by_year = df[['value', 'year', 'week']].groupby(['week', 'year'])['value'].mean().reset_index()

fig1 = px.line(df_by_year, x="week", y="value", color="year", 
    title="Comparaison consommation horaire de puissance électrique (2021-2025)",
    labels={'week': 'Semaine', 'value': 'Consommation (MW)'}
    )
fig1.show()

In [27]:
# décomposition de la date en plusieurs colonnes
df['year'] = df['end_date'].dt.year
df['month'] = df['end_date'].dt.month
df['week'] = df['end_date'].dt.isocalendar().week
df['day_of_week'] = df['end_date'].dt.dayofweek
df['hour'] = df['end_date'].dt.hour
df['day_of_year'] = df['end_date'].dt.dayofyear
df['day_of_year_align'] = (df['week'] - 1) * 7 + df['day_of_week']

### Tendance consommation par heure

In [48]:
fig_hours = px.scatter(df, x='hour', y='value', 
        title="Consommation horaire de puissance électrique en France (2021-2025) par heure de la journée",
        labels={'hour': 'Heures', 'value': 'Consommation (MW)'})
fig_hours.show()

In [ ]:
mask_day = (df['day_of_year'] == 80) & (df['year'] == 2024)
fig = px.line(df[mask_day], x='end_date', y='value', title="Consommation horaire de puissance électrique en France sur une journée",
    labels={'end_date': 'Date', 'value': 'Consommation (MW)'})
fig.show()

### Tendance consommation par jour de la semaine

In [39]:
fig_dow = px.scatter(df, x='day_of_week', y='value', color="hour",
        title="Consommation horaire de puissance électrique en France (2021-2025) par jour de la semaine",
        labels={'hour': 'Heures', 'value': 'Consommation (MW)', 'day_of_week': ''})
fig_dow.update_xaxes(
    tickvals=[0, 1, 2, 3, 4, 5, 6],
    ticktext=["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
)
fig_dow.show()

In [50]:
mask_week = (df['week'] == 13) & (df['year'] == 2024)
fig = px.line(df[mask_week], x='end_date', y='value', title="Consommation horaire de puissance électrique en France sur une semaine",
    labels={'end_date': 'Date', 'value': 'Consommation (MW)'})
fig.show()

### Tendance consommation par semaine

In [41]:
fig_hours = px.scatter(df, x='week', y='value', 
        title="Consommation horaire de puissance électrique en France (2021-2025) par semaine",
        labels={'week_of_year': 'Semaine', 'value': 'Consommation (MW)'})
fig_hours.show()

### Tendance consommation par mois

In [43]:
fig_month = px.scatter(df, x='month', y='value', 
        title="Consommation horaire de puissance électrique en France (2021-2025) par mois",
        labels={'month': 'Mois', 'value': 'Consommation (MW)'})
fig_month.update_xaxes(
    tickvals=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    ticktext=["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre"]
)
fig_month.show()